In [1]:
%pip install pennylane pennylane-lightning matplotlib scipy --break-system-packages -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 52.2 MB/s eta 0:00:00


In [4]:
import numpy as np
import pennylane as qml
from scipy.linalg import sqrtm

def build_step3_circuit(V_QSVT, V_H, qsvt_scale):
    """Combines the QSVT matrix and original block encoding into the Walk Operator."""

    # 1. Calculate weights, but split the QSVT probability mass exactly in half
    c0 = (1.0 / qsvt_scale) * np.sin(np.pi / 14)
    c1 = 1.0 * np.sin(np.pi / 14)
    c2 = 1.0 - c0 - c1

    # 2. Assign the 4 branches to perfectly utilize the 2-qubit LCU
    prep_state = np.array([
        np.sqrt(c0 / 2.0), # op0: + QSVT
        np.sqrt(c0 / 2.0), # op1: + QSVT Adjoint
        np.sqrt(c1),       # op2: V_H
        np.sqrt(c2)        # op3: Garbage collection
    ])

    wire_order = ["P0", "P1", "C", "A0", "A1", "T", "S"]
    dev = qml.device("default.qubit", wires=wire_order)

    @qml.qnode(dev)
    def step3_node():
        qml.StatePrep(prep_state, wires=["P0", "P1"])

        # Branch 0: The standard QSVT unitary (P + iQ)
        op0 = qml.prod(qml.X("T"), qml.QubitUnitary(V_QSVT, wires=["C", "A0", "A1", "S"]))

        # Branch 1: The adjoint QSVT unitary (P - iQ). The imaginary parts will annihilate!
        op1 = qml.prod(qml.X("T"), qml.adjoint(qml.QubitUnitary(V_QSVT, wires=["C", "A0", "A1", "S"])))

        # Branch 2: The original transition matrix block encoding
        op2 = qml.prod(qml.Z("T"), qml.QubitUnitary(V_H, wires=["A0", "A1", "S"]))

        # Branch 3: Dump the leftover probability mass
        op3 = qml.X("A0")

        qml.Select([op0, op1, op2, op3], control=["P0", "P1"])
        qml.adjoint(qml.StatePrep)(prep_state, wires=["P0", "P1"])

        return qml.state()

    U_Step3 = qml.matrix(step3_node, wire_order=wire_order)()

    # Extract the top-left 4x4 matrix
    final_encoded_walk = U_Step3[0:4, 0:4]

    return U_Step3, final_encoded_walk


def validate_step3(H_target, final_encoded_walk):
    """
    Validates the 4x4 matrix extracted from Step 3 against the
    analytical Szegedy Quantum Walk operator.
    """
    I = np.eye(2)
    H_squared = H_target @ H_target
    sqrt_term = sqrtm(I - H_squared)

    # Construct the exact 4x4 Walk Operator
    # Top row: [ H , sqrt ]
    # Bot row: [ sqrt, -H ]
    top_half = np.hstack([H_target, sqrt_term])
    bottom_half = np.hstack([sqrt_term, -H_target])
    walk_operator = np.vstack([top_half, bottom_half])

    # Scale by sin(pi/14) as mandated by the algorithm
    exact_target = np.sin(np.pi / 14) * walk_operator

    print("--- Quantum Circuit Output (Top-Left 4x4) ---")
    print(np.round(final_encoded_walk, 3))

    print("\n--- Exact Analytical Target ---")
    print(np.round(exact_target, 3))

    difference = exact_target - final_encoded_walk
    max_error = np.max(np.abs(difference))

    print(f"\n--- Error Magnitude ---")
    print(f"Max Element-wise Error: {max_error:.2e}")

    return max_error


import numpy as np
import pennylane as qml

np.set_printoptions(precision=3, suppress=True, linewidth=1000, threshold=np.inf)

def fmt(A, tol=5e-4):
    A = np.array(A, dtype=complex)
    A.real[np.abs(A.real) < tol] = 0
    A.imag[np.abs(A.imag) < tol] = 0
    A = np.round(A, 3)
    return np.real_if_close(A)

def show(name, A):
    print(f"\n{name}=")
    print(fmt(A))

def create_messy_VH(raw_matrix):
    H = (raw_matrix + raw_matrix.conj().T) / 2
    H = H / (np.linalg.norm(H, ord=2) + 0.1)

    dev = qml.device("default.qubit", wires=3)

    @qml.qnode(dev)
    def lazy_unitary():
        qml.BlockEncode(H, wires=[0, 1, 2])
        return qml.state()

    U_lazy = qml.matrix(lazy_unitary)()

    W_anc = np.zeros((4, 4), dtype=complex)
    W_anc[0, 0] = 1.0
    w = np.exp(2j * np.pi / 3)
    W_junk = np.array([
        [1, 1, 1],
        [1, w, w**2],
        [1, w**2, w]
    ]) / np.sqrt(3)
    W_anc[1:, 1:] = W_junk

    V_H = np.kron(W_anc, np.eye(2)) @ U_lazy
    return H, V_H

def build_lcu_encoding(V_matrix):
    wire_order = ["C", "A_0", "A_1", "S"]
    dev = qml.device("default.qubit", wires=wire_order)

    V_dagger = np.conj(V_matrix.T)
    R_mat = np.diag([-1, 1, 1, 1])

    @qml.qnode(dev)
    def circ(V_mat, V_dag):
        qml.RY(5 * np.pi / 6, wires="C")

        def apply_W():
            qml.QubitUnitary(V_mat, wires=["A_0", "A_1", "S"])
            qml.QubitUnitary(R_mat, wires=["A_0", "A_1"])
            qml.QubitUnitary(V_dag, wires=["A_0", "A_1", "S"])

        qml.ctrl(apply_W, control="C")()
        qml.RY(-np.pi / 6, wires="C")
        return qml.state()

    U_LCU = qml.matrix(circ, wire_order=wire_order)(V_matrix, V_dagger)
    return U_LCU, U_LCU[:2, :2]

def cheb_sqrt2x(deg=21, max_scale=0.5):
    if deg % 2 == 0:
        deg += 1
    f = lambda x: np.sign(x) * np.sqrt(2 * np.abs(x))
    xs = np.polynomial.chebyshev.chebpts1(2 * deg)
    ys = f(xs)
    c = np.polynomial.chebyshev.chebfit(xs, ys, deg)
    mono = np.polynomial.chebyshev.cheb2poly(c)
    mono[0::2] = 0
    poly = np.polynomial.polynomial.Polynomial(mono)
    grid = np.linspace(-1, 1, 4000)
    scale = max_scale / np.max(np.abs(poly(grid)))
    return mono * scale, scale

def qsvt_on_lcu(V_LCU, H_enc, deg=21):
    mono, scale = cheb_sqrt2x(deg)
    angles = qml.poly_to_angles(mono, "QSVT")

    dev = qml.device("default.qubit", wires=4)

    @qml.qnode(dev)
    def circ():
        U = qml.QubitUnitary(V_LCU, wires=range(4))
        P = [qml.PCPhase(float(a), dim=2, wires=range(4)) for a in angles]
        qml.QSVT(U, P)
        return qml.state()

    Q = qml.matrix(circ)()
    blk = np.real(Q[:2, :2])
    approx = blk / scale

    lam, U = np.linalg.eigh(H_enc)
    exact = U @ np.diag(np.sqrt(1 - lam**2)) @ U.T

    return {
        "Q": Q,
        "blk": blk,
        "scale": scale,
        "angles": len(angles),
        "scaled_exact": scale * exact,
        "approx": approx,
        "exact": exact,
        "valid_approx": approx / np.sqrt(8),
        "valid_target": exact / np.sqrt(8),
        "blk_err": np.max(np.abs(blk - scale * exact)),
        "valid_err": np.max(np.abs(approx / np.sqrt(8) - exact / np.sqrt(8))),
    }

raw = np.array([
    [2.0 + 1j, 3.0],
    [-1.0, 4.0 - 2j]
], dtype=complex)

H, V_H = create_messy_VH(raw)
V_LCU, lcu_block = build_lcu_encoding(V_H)

H_enc = np.real(V_H[:2, :2])
math_target = (np.eye(2) - H_enc @ H_enc) / 2.0

out = qsvt_on_lcu(V_LCU, H_enc, deg=41)

print("V_H shape =", V_H.shape)
print("V_LCU shape =", V_LCU.shape)
print("QSVT full shape =", out["Q"].shape)

show("H", H_enc)
show("VH", V_H)
show("LCU block", lcu_block)
show("Math target", math_target)
print("\nMatch =", np.allclose(lcu_block, math_target, atol=1e-6))

print("\nscale =", round(out["scale"], 6), " angles =", out["angles"])
print("block err =", f"{out['blk_err']:.2e}")
print("valid err =", f"{out['valid_err']:.2e}")

show("QSVT block", out["blk"])
show("scale*sqrt(I-H^2)", out["scaled_exact"])
show("validate approx", out["valid_approx"])
show("validate target", out["valid_target"])
show("Full LCU result", V_LCU)
show("Full QSVT matrix", out["Q"])


V_QSVT = out["Q"]
qsvt_scale = out["scale"]

# Execute Step 3 with the exact scale
U_Step3, walk_block = build_step3_circuit(V_QSVT, V_H, qsvt_scale)

np.set_printoptions(
    linewidth=150,
    precision=3,
    suppress=True,
    formatter={'complexfloat': lambda x: f"{x.real:.3f}" if np.abs(x.imag) < 1e-8 else f"{x.real:.3f}+{x.imag:.3f}j"}
)

print("\n=== STEP 3: SZEGEDY QUANTUM WALK OPERATOR ===")
max_err = validate_step3(H_enc, walk_block)
# print("\n", U_Step3)

V_H shape = (8, 8)
V_LCU shape = (16, 16)
QSVT full shape = (16, 16)

H=
[[0.393 0.196]
 [0.196 0.785]]

VH=
[[ 0.393+0.j   0.196+0.j   0.884+0.j  -0.159+0.j   0.   +0.j   0.   +0.j   0.   +0.j   0.   +0.j ]
 [ 0.196+0.j   0.785+0.j  -0.159+0.j   0.565+0.j   0.   +0.j   0.   +0.j   0.   +0.j   0.   +0.j ]
 [ 0.511+0.j  -0.092+0.j  -0.227+0.j  -0.113+0.j   0.577+0.j   0.   +0.j   0.577+0.j   0.   +0.j ]
 [-0.092+0.j   0.326+0.j  -0.113+0.j  -0.453+0.j   0.   +0.j   0.577+0.j   0.   +0.j   0.577+0.j ]
 [ 0.511+0.j  -0.092+0.j  -0.227+0.j  -0.113+0.j  -0.289+0.5j  0.   +0.j  -0.289-0.5j  0.   +0.j ]
 [-0.092+0.j   0.326+0.j  -0.113+0.j  -0.453+0.j   0.   +0.j  -0.289+0.5j  0.   +0.j  -0.289-0.5j]
 [ 0.511+0.j  -0.092+0.j  -0.227+0.j  -0.113+0.j  -0.289-0.5j  0.   +0.j  -0.289+0.5j  0.   +0.j ]
 [-0.092+0.j   0.326+0.j  -0.113+0.j  -0.453+0.j   0.   +0.j  -0.289-0.5j  0.   +0.j  -0.289+0.5j]]

LCU block=
[[ 0.404 -0.116]
 [-0.116  0.173]]

Math target=
[[ 0.404 -0.116]
 [-0.116  0.173]]

M

In [13]:
import numpy as np
from scipy.linalg import sqrtm

def good_projector(dim, good_dim = 4):
    Pi = np.zeros((dim, dim), dtype = complex)
    Pi[:good_dim, :good_dim] = np.eye(good_dim, dtype = complex)
    return Pi

def good_bad_probabilities(U, good_dim = 4):
    """
    For each logical basis input |0...0>⊗|k>, k=0..good_dim-1,
    compute:
      p_good = probability output stays in the good subspace
      p_bad  = probability output leaks outside the good subspace
    """
    good_block = U[:good_dim, :good_dim]
    bad_block = U[good_dim:, :good_dim]

    p_good = np.sum(np.abs(good_block) ** 2, axis = 0)
    p_bad = np.sum(np.abs(bad_block) ** 2, axis = 0)

    return p_good, p_bad

def random_state_good_bad_probability(U, good_dim = 4, seed = 0):
    """
    Same diagnostic, but for one random normalized input state
    supported only on the good subspace.
    """
    rng = np.random.default_rng(seed)
    psi_good = rng.normal(size = good_dim) + 1j * rng.normal(size = good_dim)
    psi_good = psi_good / np.linalg.norm(psi_good)

    psi_in = np.zeros(U.shape[0], dtype = complex)
    psi_in[:good_dim] = psi_good

    psi_out = U @ psi_in
    p_good = np.sum(np.abs(psi_out[:good_dim]) ** 2)
    p_bad = np.sum(np.abs(psi_out[good_dim:]) ** 2)

    return p_good, p_bad

def oaa_reflection(U_step3, good_dim = 4):
    r"""
    Fixed Grover/OAA iterate built from the ORIGINAL step-3 unitary:
        Q = - U R U^\dagger R
    where R = 2Π - I and Π projects onto the good subspace.
    """
    dim = U_step3.shape[0]
    Pi = good_projector(dim, good_dim)
    R = 2.0 * Pi - np.eye(dim, dtype = complex)
    Q = - U_step3 @ R @ U_step3.conj().T @ R
    return Q

def apply_oaa_rounds(U_step3, rounds = 3, good_dim = 4):
    """
    Returns:
      Q      : the fixed OAA iterate
      states : [U^(0), U^(1), ..., U^(rounds)]
               where U^(0)=U_step3 and U^(j)=Q^j U_step3
    """
    Q = oaa_reflection(U_step3, good_dim)
    states = [U_step3.copy()]

    U_current = U_step3.copy()
    for _ in range(rounds):
        U_current = Q @ U_current
        states.append(U_current)

    return Q, states

def validate_step4(H_target, amplified_U, good_dim = 4):
    """
    After 3 OAA rounds, the top-left 4x4 block should approximate
    the UNscaled walk operator.
    """
    I = np.eye(2)
    H_squared = H_target @ H_target
    sqrt_term = sqrtm(I - H_squared)

    exact_walk = np.block([
        [H_target,  sqrt_term],
        [sqrt_term, -H_target]
    ])

    amplified_block = amplified_U[:good_dim, :good_dim]
    diff = exact_walk - amplified_block
    max_err = np.max(np.abs(diff))

    print("\n--- Step 4 Top-Left 4x4 ---")
    print(np.round(amplified_block, 3))

    print("\n--- Exact UNscaled Walk Operator ---")
    print(np.round(exact_walk, 3))

    print("\n--- Step 4 Error Magnitude ---")
    print(f"Max Element-wise Error: {max_err:.2e}")

    return max_err

def report_round(U, label, good_dim = 4, seed = 0):
    p_good, p_bad = good_bad_probabilities(U, good_dim)
    rp_good, rp_bad = random_state_good_bad_probability(U, good_dim, seed = seed)

    print(f"\n=== {label} ===")
    print("Per logical basis input:")
    for k in range(good_dim):
        print(
            f"  input basis {k}: "
            f"p_good = {p_good[k]:.6f}, "
            f"p_bad = {p_bad[k]:.6f}, "
            f"total = {p_good[k] + p_bad[k]:.6f}"
        )

    print(
        f"Random good-subspace input: "
        f"p_good = {rp_good:.6f}, "
        f"p_bad = {rp_bad:.6f}, "
        f"total = {rp_good + rp_bad:.6f}"
    )

def run_step4_demo(U_Step3, H_enc, good_dim = 4):
    """
    1. Measure good/bad probability before OAA
    2. Apply OAA 3 times
    3. Measure good/bad probability after each round
    4. Validate that the final top-left 4x4 is the unscaled walk
    """
    Q, rounds = apply_oaa_rounds(U_Step3, rounds = 3, good_dim = good_dim)

    report_round(rounds[0], "Before OAA (round 0)", good_dim)
    report_round(rounds[1], "After OAA round 1", good_dim)
    report_round(rounds[2], "After OAA round 2", good_dim)
    report_round(rounds[3], "After OAA round 3", good_dim)

    print("\nExpected ideal success probabilities by round:")
    for j in range(4):
        theta = np.pi / 14
        ideal = np.sin((2 * j + 1) * theta) ** 2
        print(f"  round {j}: sin^2((2*{j}+1)π/14) = {ideal:.6f}")

    validate_step4(H_enc, rounds[3], good_dim = good_dim)

    return Q, rounds


In [14]:
print("\n=== STEP 4: OAA x 3 ===")
Q_oaa, oaa_rounds = run_step4_demo(U_Step3, H_enc, good_dim = 4)
U_after_oaa = oaa_rounds[3]


=== STEP 4: OAA x 3 ===

=== Before OAA (round 0) ===
Per logical basis input:
  input basis 0: p_good = 0.049188, p_bad = 0.950812, total = 1.000000
  input basis 1: p_good = 0.049233, p_bad = 0.950767, total = 1.000000
  input basis 2: p_good = 0.049188, p_bad = 0.950812, total = 1.000000
  input basis 3: p_good = 0.049233, p_bad = 0.950767, total = 1.000000
Random good-subspace input: p_good = 0.049216, p_bad = 0.950784, total = 1.000000

=== After OAA round 1 ===
Per logical basis input:
  input basis 0: p_good = 0.386528, p_bad = 0.613472, total = 1.000000
  input basis 1: p_good = 0.386836, p_bad = 0.613164, total = 1.000000
  input basis 2: p_good = 0.386528, p_bad = 0.613472, total = 1.000000
  input basis 3: p_good = 0.386836, p_bad = 0.613164, total = 1.000000
Random good-subspace input: p_good = 0.386719, p_bad = 0.613281, total = 1.000000

=== After OAA round 2 ===
Per logical basis input:
  input basis 0: p_good = 0.808778, p_bad = 0.191222, total = 1.000000
  input basis